# Matsya goldfish detector: open-data training notebook

Trains a single-class goldfish detector from **Open Images V7** (human-drawn boxes; annotations CC BY 4.0, images listed by Google as CC BY 2.0) and exports it to ONNX for the Matsya app.
No private data, no account tokens, no paid service. Works on **Google Colab (GPU runtime)** and **Kaggle Notebooks (GPU + Internet on)**.

How to run
1. Runtime / Accelerator: **GPU (T4 or better)**. On Kaggle also switch **Internet = On**.
2. First run with `SMOKE = True` (about 10 minutes, tiny data, 1 epoch). It must finish and print `SMOKE OK`. This proves the whole pipeline.
3. Set `SMOKE = False` and use *Restart and run all*. Full run: about 25 to 50 minutes on a T4.
4. Download `matsya_detector_bundle.zip` (Colab downloads it automatically; on Kaggle it is in the Output tab).

Notes
* This model is trained on photos; it is **not validated on tank video**. Validation numbers printed here are on Open Images validation photos and are **not** the project's acceptance metric. The acceptance test runs separately on human-labelled video frames.
* Licences: Ultralytics training code and pretrained starting weights are AGPL-3.0; the resulting model must be published under AGPL-3.0 terms. Open Images attribution is in the manifest.

In [ ]:
import os, sys, json, random, shutil, zipfile, platform, time, subprocess
from pathlib import Path

SMOKE = True   # <- run once with True, then set to False and run everything again

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE:
    WORK = Path("/kaggle/working") / "matsya_train"
elif ON_COLAB:
    WORK = Path("/content") / "matsya_train"
else:
    # Local CPU smoke runs (for example, the Box) have no /content mount.
    WORK = Path(os.environ.get("MATSYA_TRAIN_WORK", "/tmp/matsya_train")).expanduser()
WORK.mkdir(parents=True, exist_ok=True)

CFG = dict(
    n_goldfish_train = 80   if SMOKE else 1500,
    n_goldfish_val   = 40   if SMOKE else 250,
    n_fish_train     = 60   if SMOKE else 1200,   # generic "Fish" class, merged into the same single class for candidate B
    n_fish_val       = 30   if SMOKE else 200,
    epochs           = 1    if SMOKE else 80,
    imgsz            = 320  if SMOKE else 640,
    batch            = 8    if SMOKE else 16,
    seed             = 0,
    glare_copy_frac  = 0.2,                       # fraction of train images that also get a glare/bubble copy
    min_train_images = 5    if SMOKE else 400,
)
print("SMOKE =", SMOKE, "| platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "other")
print(json.dumps(CFG, indent=2))

In [ ]:
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    pip("ultralytics==8.4.213")
except subprocess.CalledProcessError:
    print("pinned version unavailable, installing latest ultralytics")
    pip("ultralytics")
pip("fiftyone", "onnx", "onnxslim", "onnxruntime")
import ultralytics, torch, cv2, numpy as np
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Training will be very slow. Change the runtime type to GPU.")
else:
    print("GPU:", torch.cuda.get_device_name(0))
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

In [ ]:
import re
import numpy as np, cv2

def _flag(det, name):
    """Open Images flags (IsGroupOf, IsDepiction, ...): 1 = yes, 0 = no, -1 = unknown.
    They can be dynamic fields or Attribute objects, and may arrive as bool, int, float or string.
    Only an explicit 1 / true counts as yes (a plain bool() would treat the strings "0" and "-1" as True)."""
    v = getattr(det, name, None)
    if v is None:
        attrs = getattr(det, "attributes", None) or {}
        try:
            v = attrs.get(name)
        except Exception:
            v = None
        if hasattr(v, "value"):
            v = v.value
    if isinstance(v, str):
        return v.strip().lower() in ("1", "true", "yes")
    try:
        return float(v) == 1.0
    except (TypeError, ValueError):
        return False

REASONS = {}
def _skip(reason):
    REASONS[reason] = REASONS.get(reason, 0) + 1
    return None

def sample_to_yolo_lines(sample, target_label, field="ground_truth", min_area=0.0004):
    """Return YOLO label lines (class 0) for one Open Images sample, or None to skip the image.
    The whole image is skipped when it holds a group box or a depiction (drawing, toy) of the target class,
    because its other fish would be unlabelled and teach the model to ignore fish."""
    dets = getattr(sample, field, None)
    if dets is None:
        return _skip("no detections field")
    lines = []
    for d in dets.detections:
        if d.label != target_label:
            continue
        if _flag(d, "IsGroupOf") or _flag(d, "IsDepiction"):
            return _skip("group box or depiction")
        x, y, w, h = [float(v) for v in d.bounding_box]          # relative, top-left origin
        x0, y0 = max(0.0, x), max(0.0, y)
        x1, y1 = min(1.0, x + w), min(1.0, y + h)
        w, h = x1 - x0, y1 - y0
        if w <= 0 or h <= 0 or w * h < min_area:
            continue
        lines.append(f"0 {(x0 + w / 2):.6f} {(y0 + h / 2):.6f} {w:.6f} {h:.6f}")
    return lines if lines else _skip("no usable box")

def add_glare_bubbles(img, rng):
    """Draw soft white glare ellipses and small bubble rings. Boxes stay valid (nothing moves)."""
    h, w = img.shape[:2]
    out = img.copy()
    overlay = out.copy()
    for _ in range(int(rng.integers(1, 3))):
        cx, cy = int(rng.integers(0, w)), int(rng.integers(0, h))
        ax, ay = int(rng.integers(max(2, w // 20), max(3, w // 6))), int(rng.integers(max(2, h // 30), max(3, h // 8)))
        cv2.ellipse(overlay, (cx, cy), (ax, ay), int(rng.integers(0, 180)), 0, 360, (255, 255, 255), -1)
    overlay = cv2.GaussianBlur(overlay, (0, 0), max(1.0, w / 120))
    out = cv2.addWeighted(overlay, 0.55, out, 0.45, 0)
    for _ in range(int(rng.integers(0, 25))):
        x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
        r = int(rng.integers(2, max(4, w // 60)))
        cv2.circle(out, (x, y), r, (235, 240, 245), 1, cv2.LINE_AA)
        cv2.circle(out, (x - r // 3, y - r // 3), max(1, r // 4), (255, 255, 255), -1, cv2.LINE_AA)
    return out

def validate_yolo_dir(root, min_train):
    """Hard checks before training. Raises AssertionError on any problem."""
    root = Path(root)
    names = {}
    for split in ("train", "val"):
        imgs = sorted((root / "images" / split).glob("*.*"))
        lbls = {p.stem: p for p in (root / "labels" / split).glob("*.txt")}
        assert imgs, f"no images in {split} (skip reasons so far: {REASONS})"
        boxes = 0
        for im in imgs:
            assert im.stem in lbls, f"missing label for {im.name}"
            for ln in lbls[im.stem].read_text().strip().splitlines():
                c, cx, cy, w, h = ln.split()
                c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
                assert c == 0, f"bad class in {im.name}"
                assert 0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1, f"box out of range in {im.name}"
                boxes += 1
        names[split] = {re.sub(r"_gl$", "", re.sub(r"^(gf|fi)_", "", im.stem)) for im in imgs}
        print(f"  {split}: {len(imgs)} images, {boxes} boxes")
    assert not (names["train"] & names["val"]), "an image appears in both train and val"
    assert len(names["train"]) >= min_train, f"too few train images ({len(names['train'])} < {min_train}); skip reasons: {REASONS}"
    return {k: len(v) for k, v in names.items()}

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def load_oi(split, cls, n):
    print(f"downloading Open Images V7 {split} / {cls} (max {n}) ...")
    return foz.load_zoo_dataset(
        "open-images-v7", split=split, label_types=["detections"], classes=[cls],
        max_samples=n, shuffle=True, seed=CFG["seed"],
        dataset_name=f"oi_{split}_{cls}_{int(time.time())}",
    )

oi = {
    ("train", "Goldfish"):      load_oi("train", "Goldfish", CFG["n_goldfish_train"]),
    ("validation", "Goldfish"): load_oi("validation", "Goldfish", CFG["n_goldfish_val"]),
    ("train", "Fish"):          load_oi("train", "Fish", CFG["n_fish_train"]),
    ("validation", "Fish"):     load_oi("validation", "Fish", CFG["n_fish_val"]),
}
for k, ds in oi.items():
    print(k, len(ds), "images")
# show the real structure of one detection, so a field-name or type mismatch is visible in the log
_s0 = next(iter(oi[("train", "Goldfish")]))
_d0 = _s0.ground_truth.detections[0]
print("example detection:", {k: _d0[k] for k in _d0.field_names if k not in ("id", "tags")} if hasattr(_d0, "field_names") else _d0)

In [ ]:
MANIFEST = []

def write_split(ds, label, out, split, prefix, skip_ids=frozenset()):
    img_dir = out / "images" / split; lbl_dir = out / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    kept = 0
    for s in ds:
        src = Path(s.filepath)
        if src.stem in skip_ids:      # same photo already labelled under the other class: never add it twice
            continue
        lines = sample_to_yolo_lines(s, label)
        if not lines:
            continue
        name = f"{prefix}_{src.stem}"
        shutil.copy(src, img_dir / f"{name}{src.suffix.lower()}")
        (lbl_dir / f"{name}.txt").write_text("\n".join(lines) + "\n")
        MANIFEST.append((name, split, label, "open-images-v7"))
        kept += 1
    print(f"  {prefix}/{split}: kept {kept} of {len(ds)} images")

def make_glare_copies(out, frac, seed):
    rng = np.random.default_rng(seed)
    imgs = sorted((out / "images" / "train").glob("*.*"))
    chosen = [p for p in imgs if "_gl" not in p.stem]
    rng.shuffle(chosen)
    n = 0
    for p in chosen[: int(len(chosen) * frac)]:
        im = cv2.imread(str(p))
        if im is None:
            continue
        cv2.imwrite(str(out / "images" / "train" / f"{p.stem}_gl{p.suffix}"), add_glare_bubbles(im, rng))
        shutil.copy(out / "labels" / "train" / f"{p.stem}.txt", out / "labels" / "train" / f"{p.stem}_gl.txt")
        n += 1
    print(f"  added {n} glare/bubble copies")

DATA = WORK / "data"
if DATA.exists():
    shutil.rmtree(DATA)
A, B = DATA / "A_goldfish", DATA / "B_goldfish_plus_fish"

print("candidate A: goldfish only")
write_split(oi[("train", "Goldfish")], "Goldfish", A, "train", "gf")
write_split(oi[("validation", "Goldfish")], "Goldfish", A, "val", "gf")
shutil.copytree(A, B)
print("candidate B: goldfish + generic fish (all mapped to one class)")
def ids_in(root, split):
    return {re.sub(r"^(gf|fi)_", "", p.stem) for p in (root / "images" / split).glob("*.*")}
# also skip any Goldfish photo that was rejected (group box / depiction): its fish would otherwise appear unlabelled
rejected = {split: {Path(s.filepath).stem for s in oi[(sp, "Goldfish")] if sample_to_yolo_lines(s, "Goldfish") is None}
            for split, sp in (("train", "train"), ("val", "validation"))}
write_split(oi[("train", "Fish")], "Fish", B, "train", "fi", ids_in(B, "train") | rejected["train"])
write_split(oi[("validation", "Fish")], "Fish", B, "val", "fi", ids_in(B, "val") | rejected["val"])
print("skip reasons:", REASONS)
for root in (A, B):
    make_glare_copies(root, CFG["glare_copy_frac"], CFG["seed"])
    (root / "data.yaml").write_text(f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: goldfish\n")
    print("validating", root.name)
    try:
        # Candidate B remains subject to the hard v5 400-image floor.
        # Candidate A is secondary if Goldfish-only filtering yields fewer than 400.
        validate_yolo_dir(root, CFG["min_train_images"] if root.name.startswith("B_") else 5)
    except AssertionError as e:
        msg = f"DATASET VALIDATION FAILED for {root.name}: {e}"
        (WORK / "last_error.txt").write_text(msg)
        print(msg)          # printed in plain text so it can be copied from the log
        raise

with open(WORK / "dataset_manifest.csv", "w") as f:
    f.write("image,split,source_class,source\n")
    for r in MANIFEST:
        f.write(",".join(r) + "\n")
print("manifest rows:", len(MANIFEST))

In [ ]:
from ultralytics import YOLO
RUNS = WORK / "runs"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
t0 = time.time()

def train_candidate(name, data_yaml, base):
    m = YOLO(base)
    m.train(data=str(data_yaml), epochs=CFG["epochs"], imgsz=CFG["imgsz"], batch=CFG["batch"],
            patience=15, seed=CFG["seed"], deterministic=True, project=str(RUNS), name=name,
            exist_ok=True, workers=2, plots=False, device=DEVICE,
            mosaic=1.0, close_mosaic=10, degrees=5.0, scale=0.5, fliplr=0.5, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4)
    return Path(m.trainer.save_dir) / "weights" / "best.pt"

best = {}
best["A_yolov8n_goldfish"] = train_candidate("A_yolov8n_goldfish", A / "data.yaml", "yolov8n.pt")
best["B_yolov8s_goldfish_plus_fish"] = train_candidate("B_yolov8s_goldfish_plus_fish", B / "data.yaml", "yolov8s.pt")
print("training seconds:", round(time.time() - t0))

In [ ]:
def export_and_check(name, weights):
    sz = CFG["imgsz"]
    onnx_path = Path(YOLO(str(weights)).export(format="onnx", opset=12, simplify=True, imgsz=sz, dynamic=False))
    net = cv2.dnn.readNetFromONNX(str(onnx_path))
    blob = cv2.dnn.blobFromImage(np.full((sz, sz, 3), 114, np.uint8), 1 / 255.0, (sz, sz), swapRB=True)
    net.setInput(blob)
    out = net.forward()
    # The Matsya loader decodes the standard layout [1, 4+nc, N] (here nc=1 -> 5 rows). Anything else must not be shipped.
    assert out.ndim == 3 and out.shape[1] == 5, f"unexpected ONNX output layout {out.shape}"
    print(name, "ONNX OK, output shape", out.shape, "->", onnx_path.name)
    meta = {"input": sz, "classes": ["goldfish"], "conf": 0.25, "iou": 0.5, "version": name,
            "trained_on": "open-images-v7 (human-drawn boxes) + synthetic glare/bubble copies",
            "eval": None, "note": "acceptance evaluation on human-labelled video frames is run separately"}
    meta_path = onnx_path.with_suffix(".json")
    meta_path.write_text(json.dumps(meta, indent=2))
    return onnx_path, meta_path

exported = {n: export_and_check(n, w) for n, w in best.items()}

In [ ]:
rows = []
for name, w in best.items():
    d = A if name.startswith("A_") else B
    r = YOLO(str(w)).val(data=str(d / "data.yaml"), imgsz=CFG["imgsz"], batch=CFG["batch"], plots=False, device=DEVICE, verbose=False)
    rows.append((name, round(float(r.box.mp), 3), round(float(r.box.mr), 3), round(float(r.box.map50), 3), round(float(r.box.map), 3)))
print("In-distribution check on Open Images validation PHOTOS (NOT the acceptance metric):")
print("candidate, precision, recall, mAP50, mAP50-95")
for r in rows:
    print(*r, sep=", ")

In [ ]:
env = {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "smoke": SMOKE, "cfg": CFG, "train_seconds": round(time.time() - t0)}
(WORK / "env.json").write_text(json.dumps(env, indent=2))
(WORK / "in_distribution_val.csv").write_text("candidate,precision,recall,map50,map50_95\n" + "\n".join(",".join(map(str, r)) for r in rows) + "\n")

zip_path = WORK / ("matsya_detector_bundle_SMOKE.zip" if SMOKE else "matsya_detector_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name, w in best.items():
        z.write(w, f"{name}/best.pt")
        z.write(Path(w).parent.parent / "results.csv", f"{name}/results.csv")
        z.write(Path(w).parent.parent / "args.yaml", f"{name}/args.yaml")
        o, m = exported[name]
        z.write(o, f"{name}/goldfish.onnx"); z.write(m, f"{name}/goldfish.json")
    for f in ("dataset_manifest.csv", "env.json", "in_distribution_val.csv"):
        z.write(WORK / f, f)
print("bundle:", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")
if SMOKE:
    print("SMOKE OK  -> now set SMOKE = False and Restart and run all")
if ON_COLAB:
    from google.colab import files
    files.download(str(zip_path))
elif ON_KAGGLE:
    shutil.copy(zip_path, "/kaggle/working/" + zip_path.name)
    shutil.rmtree(DATA, ignore_errors=True)       # keep the Output tab small: the images are not needed after training
    shutil.rmtree(RUNS, ignore_errors=True)
    print("Kaggle: find", zip_path.name, "in the Output tab")